In [ ]:
[ ]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load the datasets
pokemon_df = pd.read_csv("pokemon.csv")
combats_df = pd.read_csv("combats.csv")

# 1. Fix missing Name for Pokemon #62
pokemon_df.loc[pokemon_df["#"] == 62, "Name"] = "Primeape"

# 2. Handle NaN values in Type 2
pokemon_df["Type 2"] = pokemon_df["Type 2"].fillna("None")

# 3. Calculate Win Percentage
# Count total fights for First_pokemon and Second_pokemon
first_fights = combats_df["First_pokemon"].value_counts()
second_fights = combats_df["Second_pokemon"].value_counts()
total_fights = first_fights.add(second_fights, fill_value=0)

# Count total wins
wins = combats_df["Winner"].value_counts()

# Merge fight statistics into pokemon_df
pokemon_df["Total_Fights"] = (
    pokemon_df["#"].map(total_fights).fillna(0).astype(int)
)
pokemon_df["Wins"] = pokemon_df["#"].map(wins).fillna(0).astype(int)
pokemon_df["Win_Percentage"] = (
    pokemon_df["Wins"] / pokemon_df["Total_Fights"]
) * 100

# Fill any NaN in Win_Percentage (for Pokemon that never fought) with 0
pokemon_df["Win_Percentage"] = pokemon_df["Win_Percentage"].fillna(0)

print("Data Preparation Complete!")
display(pokemon_df.head())

In [ ]:
[ ]
# Define numerical stats columns
stats_cols = [
    "HP",
    "Attack",
    "Defense",
    "Sp. Atk",
    "Sp. Def",
    "Speed",
    "Win_Percentage",
]

# 1. Correlation Matrix
plt.figure(figsize=(10, 8))
corr_matrix = pokemon_df[stats_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix: Pokemon Stats vs. Win Percentage")
plt.show()

# 2. Pairplot of key stats vs. Win Percentage
sns.pairplot(
    pokemon_df[["HP", "Attack", "Speed", "Win_Percentage"]],
    diag_kind="kde",
    plot_kws={"alpha": 0.5},
)
plt.suptitle(
    "Pairplot of HP, Attack, Speed vs Win Percentage", y=1.02, fontsize=14
)
plt.show()

# 3. Top 10 Pokemon by Win Percentage
top_10 = pokemon_df.sort_values(by="Win_Percentage", ascending=False).head(10)
print("\n--- TOP 10 POKEMON BY WIN PERCENTAGE ---")
display(
    top_10[["#", "Name", "Type 1", "Type 2", "Speed", "Attack", "Win_Percentage"]]
)

# Barplot of Top 10
plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_10,
    x="Win_Percentage",
    y="Name",
    hue="Type 1",
    dodge=False,
    palette="viridis",
)
plt.title("Top 10 Pokemon by Win Percentage")
plt.xlabel("Win Percentage (%)")
plt.ylabel("Pokemon Name")
plt.legend(title="Primary Type", loc="lower right")
plt.show()

In [ ]:
[ ]
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Select features and target variable
feature_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = pokemon_df[feature_cols]
y = pokemon_df["Win_Percentage"]

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(
        n_estimators=100, learning_rate=0.05, random_state=42
    ),
}

# Train and Evaluate Models
results = []

for name, model in models.items():
  model.fit(X_train, y_train)
  preds = model.predict(X_test)

  mae = mean_absolute_error(y_test, preds)
  rmse = np.sqrt(mean_squared_error(y_test, preds))
  r2 = r2_score(y_test, preds)

  results.append(
      {"Model": name, "MAE (Mean Absolute Error)": mae, "RMSE": rmse, "R2 Score": r2}
  )

# Display Model Comparison
results_df = pd.DataFrame(results).sort_values(
    by="MAE (Mean Absolute Error)"
)
print("--- MODEL PERFORMANCE COMPARISON ---")
display(results_df)

# Visualize MAE comparison
plt.figure(figsize=(8, 5))
sns.barplot(
    data=results_df,
    x="Model",
    y="MAE (Mean Absolute Error)",
    palette="Blues_d",
)
plt.title("Model Comparison: Mean Absolute Error (Lower is better)")
plt.ylabel("MAE (% Win Rate Error)")
plt.show()